# Create Breakthrough Prize Awards (PRIZE PATTERN)

Breakthrough Prize laureates from the official static HTML pages on `breakthroughprize.org`.

**Prerequisite:** run `scripts/local/breakthrough_prize_to_s3.py` to fetch the official listing/detail pages and upload parquet to S3.

**Source:** `https://breakthroughprize.org/Laureates/1`, plus discovered category/prize/year/detail pages for Fundamental Physics, Life Sciences, and Mathematics.

**S3 location:** `s3://openalex-ingest/awards/breakthrough_prize/breakthrough_prize_laureates.parquet`

**Awarding body:** Breakthrough Prize Foundation - F4320315036

Funder details from OpenAlex Step 0:

- funder_id: `4320315036`
- display_name: `Breakthrough Prize Foundation`
- ror_id: NULL in OpenAlex API result
- doi: `10.13039/100014397`

Prize-pattern notes:

- `lead_investigator` is the laureate. Some official laureates are collaborations or institutions; those names are preserved as the source publishes them.
- `funder_scheme` is the official prize title from each detail page.
- Official prize pages publish USD amounts for active prize types. The source script apportions the prize amount equally across each official listing-page winner group.
- The 10 discontinued `Physics Frontiers Prize in Fundamental Physics` rows have NULL amount/currency because the current official prize pages do not publish an amount for that prize type. This is the only Step 6.7 amount waiver in this ingest.
- `funder_award_id` is a synthetic key built from prize title, year, and official laureate detail ID. The source script hard-fails on duplicate keys.


## Step 1: Create Raw Table


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.breakthrough_prize_raw
USING delta AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/breakthrough_prize/breakthrough_prize_laureates.parquet`;


## Step 1.5: Inspect Raw Data First

Verify exact source columns and values before relying on the transform below.


In [ ]:
%sql
DESCRIBE openalex.awards.breakthrough_prize_raw;


In [ ]:
%sql
SELECT *
FROM openalex.awards.breakthrough_prize_raw
LIMIT 10;


In [ ]:
%sql
SELECT
  prize_title,
  COUNT(*) AS rows,
  COUNT(amount_usd) AS rows_with_amount,
  collect_set(currency) AS currencies,
  MIN(TRY_CAST(amount_usd AS DOUBLE)) AS min_amount,
  MAX(TRY_CAST(amount_usd AS DOUBLE)) AS max_amount,
  AVG(TRY_CAST(amount_usd AS DOUBLE)) AS avg_amount
FROM openalex.awards.breakthrough_prize_raw
GROUP BY prize_title
ORDER BY prize_title;


In [ ]:
%sql
SELECT
  prize_title,
  award_year,
  listing_path,
  listing_group_index,
  listing_group_size,
  award_group_size,
  portion,
  amount_usd,
  laureate_name
FROM openalex.awards.breakthrough_prize_raw
ORDER BY TRY_CAST(award_year AS INT) DESC, prize_title, listing_group_index, laureate_name
LIMIT 50;


## Step 1.6: Funder Existence Fail-Fast

Must return exactly one row. If zero rows, stop before transforming because the CROSS JOIN would emit zero awards.


In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320315036;


## Step 2: Transform To OpenAlex Award Schema


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.breakthrough_prize_awards
USING delta
AS
WITH breakthrough_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320315036
), normalized AS (
    SELECT
        *,
        TRY_CAST(award_year AS INT) AS award_year_int,
        TRY_CAST(amount_usd AS DOUBLE) AS amount_double,
        NULLIF(TRIM(currency), '') AS currency_norm,
        NULLIF(TRIM(prize_title), '') AS prize_title_norm,
        NULLIF(TRIM(citation), '') AS citation_norm,
        NULLIF(TRIM(affiliation), '') AS affiliation_norm,
        NULLIF(TRIM(laureate_given_name), '') AS given_name_norm,
        NULLIF(TRIM(laureate_family_name), '') AS family_name_norm,
        NULLIF(TRIM(laureate_name), '') AS laureate_name_norm,
        NULLIF(TRIM(detail_path), '') AS detail_url_norm
    FROM openalex.awards.breakthrough_prize_raw
)
SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':breakthrough:', LOWER(n.funder_award_id)))) % 9000000000 AS id,
    CONCAT(CAST(n.award_year_int AS STRING), ' ', n.prize_title_norm, ' - ', n.laureate_name_norm) AS display_name,
    CASE
        WHEN TRY_CAST(n.declined AS BOOLEAN) AND n.citation_norm IS NOT NULL
            THEN CONCAT('Declined the prize. ', n.citation_norm)
        WHEN TRY_CAST(n.declined AS BOOLEAN) THEN 'Declined the prize.'
        ELSE n.citation_norm
    END AS description,
    f.funder_id,
    n.funder_award_id,
    n.amount_double AS amount,
    n.currency_norm AS currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) AS id,
        f.display_name, f.ror_id, f.doi
    ) AS funder,
    'prize' AS funding_type,
    n.prize_title_norm AS funder_scheme,
    'breakthrough_prize' AS provenance,
    TRY_TO_DATE(CONCAT(CAST(n.award_year_int AS STRING), '-01-01'), 'yyyy-MM-dd') AS start_date,
    TRY_TO_DATE(CONCAT(CAST(n.award_year_int AS STRING), '-12-31'), 'yyyy-MM-dd') AS end_date,
    n.award_year_int AS start_year,
    n.award_year_int AS end_year,
    struct(
        n.given_name_norm AS given_name,
        n.family_name_norm AS family_name,
        CAST(NULL AS STRING) AS orcid,
        CAST(NULL AS DATE) AS role_start,
        struct(
            n.affiliation_norm AS name,
            CAST(NULL AS STRING) AS country,
            CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
        ) AS affiliation
    ) AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.detail_url_norm AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':breakthrough:', LOWER(n.funder_award_id)))) % 9000000000) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM normalized n
CROSS JOIN breakthrough_funder f
WHERE n.funder_award_id IS NOT NULL
  AND n.award_year_int IS NOT NULL
  AND n.prize_title_norm IS NOT NULL
  AND n.laureate_name_norm IS NOT NULL;


## Step 3: Insert Into openalex_awards_raw At Priority 63


In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'breakthrough_prize' AND priority = 63;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT id, display_name, description, funder_id, funder_award_id,
       amount, currency, funder, funding_type, funder_scheme, provenance,
       start_date, end_date, start_year, end_year,
       lead_investigator, co_lead_investigator, investigators,
       landing_page_url, doi, works_api_url,
       created_date, updated_date,
       63 AS priority
FROM openalex.awards.breakthrough_prize_awards;


## Verification


In [ ]:
%sql
SELECT COUNT(*) AS total
FROM openalex.awards.breakthrough_prize_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.breakthrough_prize_awards;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_lead_investigator,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(start_date), COUNT(*)) * 100.0, 1) AS pct_dates
FROM openalex.awards.breakthrough_prize_awards;


In [ ]:
%sql
SELECT funder.display_name, funder_id, COUNT(*) AS rows
FROM openalex.awards.breakthrough_prize_awards
GROUP BY funder.display_name, funder_id;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount,
    AVG(amount) AS avg_amount
FROM openalex.awards.breakthrough_prize_awards;


In [ ]:
%sql
-- Expected: only Physics Frontiers rows should have NULL amount/currency.
SELECT funder_scheme, COUNT(*) AS rows_without_amount
FROM openalex.awards.breakthrough_prize_awards
WHERE amount IS NULL OR currency IS NULL
GROUP BY funder_scheme
ORDER BY rows_without_amount DESC;


In [ ]:
%sql
SELECT start_year, funder_scheme, COUNT(*) AS rows
FROM openalex.awards.breakthrough_prize_awards
GROUP BY start_year, funder_scheme
ORDER BY start_year DESC, funder_scheme;


In [ ]:
%sql
SELECT
    id,
    SUBSTRING(display_name, 1, 100) AS display_name,
    funder_scheme,
    amount,
    currency,
    start_year,
    lead_investigator.given_name,
    lead_investigator.family_name,
    lead_investigator.affiliation.name AS affiliation,
    landing_page_url
FROM openalex.awards.breakthrough_prize_awards
ORDER BY start_year DESC, funder_scheme, display_name
LIMIT 25;


In [ ]:
%sql
-- ID/funder_award_id uniqueness guard after transformation.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT id) AS distinct_openalex_ids,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.breakthrough_prize_awards;
